# In-Memory Vector Store

## 1. What is an In-Memory Vector Store?

An **In-Memory Vector Store** is a data structure that stores **vector embeddings** entirely in **RAM** instead of disk or an external database.  
It allows fast **similarity search** (semantic search) using techniques like cosine similarity, dot product, or Euclidean distance.

Each stored item typically contains:
- **ID** – unique identifier
- **Vector** – embedding (e.g., 384 / 768 / 1536 dimensions)
- **Text** – original chunk or content
- **Metadata** – document name, page number, tags, etc.

Because everything lives in memory:
- Retrieval is **very fast**
- Data is **lost on application restart** unless explicitly saved

---

## 2. Core Functions

An in-memory vector store usually supports:

1. **Upsert**
   - Add or update vectors
2. **Search**
   - Find top-K most similar vectors to a query
3. **Filter (optional)**
   - Filter results using metadata
4. **Delete (optional)**
   - Remove vectors by ID

---

## 3. Why Use In-Memory?

### Advantages
- Extremely fast (RAM access)
- Zero infrastructure setup
- Perfect for prototyping
- Easy to debug
- No network latency

### Limitations
- No persistence by default
- Limited by available RAM
- Not ideal for large-scale or multi-user systems

---

## 4. Common Use Cases

### 4.1 RAG Prototypes
Ideal for early-stage **Retrieval-Augmented Generation** apps where document size is small.

### 4.2 Session-Based Retrieval
Store vectors only for the duration of a user session (e.g., uploaded PDFs).

### 4.3 Testing & CI
Useful for unit tests without requiring a vector database.

### 4.4 Edge or Offline Systems
Works where external databases are restricted or unavailable.

### 4.5 Embedding Cache
Store frequently used embeddings to avoid recomputation.

---

## 5. Simple Architecture

```text
Documents
   ↓
Chunking
   ↓
Embedding Model
   ↓
In-Memory Vector Store (RAM)
   ↓
Similarity Search
   ↓
LLM / Application Logic


## 6. Similarity Methods

| Method            | Notes                                    |
| ----------------- | ---------------------------------------- |
| Cosine Similarity | Most common for text embeddings          |
| Dot Product       | Fast, works well with normalized vectors |
| Euclidean (L2)    | Often used in FAISS                      |


## 07. When NOT to Use In-Memory

* Avoid in-memory vector stores when:
* You need restart-safe persistence
* You have >100k vectors
* You need concurrent multi-user access
* You need advanced filtering & hybrid search

In those cases, use:

* Chroma
* pgvector
* Pinecone
* Weaviate
* Milvus

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

from langchain.chat_models import init_chat_model

llm=init_chat_model("openai:gpt-4o-mini")
llm


ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000019AB76347D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000019AB7656B10>, root_client=<openai.OpenAI object at 0x0000019AB5A4F290>, root_async_client=<openai.AsyncOpenAI object at 0x0000019AB60F32F0>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [2]:
from langchain_openai import OpenAIEmbeddings

from langchain_core.vectorstores import InMemoryVectorStore

vector_store=InMemoryVectorStore(embedding=OpenAIEmbeddings())

In [3]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [4]:
vector_store.add_documents(documents=documents)

['2e4a4481-fecc-4bc5-ac64-fb349a283bc1',
 '22cc6bc9-0ed2-4750-b386-475c93120e0e',
 'a38fb70c-a505-4108-b98c-38281f50e7ce',
 'f96731f6-11df-46bd-8064-6fb141ce7005',
 '4b100920-e6d3-4720-8cd6-9f0bcc39f9bc',
 '64ab43fb-411c-4a8f-8b4c-ce9ff63c9b8a',
 '2b4ef7da-2484-4de9-9160-efd97bb01891',
 'feeb2ab7-5c03-4195-acf0-dd6fe940f850',
 'fe21ba4b-3c8d-4573-bb26-755791e7a55a',
 'a3c2fd9d-6c4c-41d9-8959-4242ee8dea93']

In [5]:
vector_store.similarity_search("hows the weather forecast")

[Document(id='22cc6bc9-0ed2-4750-b386-475c93120e0e', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='fe21ba4b-3c8d-4573-bb26-755791e7a55a', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='a3c2fd9d-6c4c-41d9-8959-4242ee8dea93', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='2b4ef7da-2484-4de9-9160-efd97bb01891', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

In [6]:
vector_store.similarity_search("hows the weather forecast",k=2)

[Document(id='22cc6bc9-0ed2-4750-b386-475c93120e0e', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='fe21ba4b-3c8d-4573-bb26-755791e7a55a', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.')]

In [7]:
### vectorstore to retriever

retriever=vector_store.as_retriever(search_kwargs={"k":2})

retriever

VectorStoreRetriever(tags=['InMemoryVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x0000019AB768FF50>, search_kwargs={'k': 2})

In [8]:
## Invoke
retriever.invoke("hows the weather forecast")

[Document(id='22cc6bc9-0ed2-4750-b386-475c93120e0e', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='fe21ba4b-3c8d-4573-bb26-755791e7a55a', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.')]